# Pipeline de Demanda Odontológica por Bairro - Sprint 4 IA

**Projeto:** Tech do Bem — FIAP — 1TDSPR

Notebook do pipeline de classificação multi-classe (Baixa / Media / Alta) da demanda odontológica por bairro de São Paulo.

Etapas:
1. Leitura do dataset sintético
2. EDA + visualizações
3. Split estratificado treino/teste
4. Treino de 3 modelos com validação cruzada 5-fold estratificada
5. Seleção pelo `f1_macro`
6. Avaliação final no conjunto de teste
7. Feature importances (incluindo a decoy `dia_semana_cadastro`)
8. Serialização do pipeline vencedor em `.joblib` + `metrics.json`

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
    XGB_OK = True
except Exception:
    XGB_OK = False
    from sklearn.ensemble import GradientBoostingClassifier

SEED = 42
ROOT = Path('..').resolve()
CSV_PATH = ROOT / 'data' / 'demanda_bairros.csv'
print('xgboost disponivel:', XGB_OK)

## 1. Leitura do dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
print('shape:', df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
df['demanda'].value_counts().sort_index()

## 2. EDA - Visualizações

In [ ]:
features = [
    'densidade_pop', 'idh', 'dist_ubs_km', 'historico_atendimentos',
    'pct_exames_pendentes', 'populacao_infantil_pct', 'dia_semana_cadastro',
]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, features):
    ax.hist(df[col], bins=30)
    ax.set_title(col)
axes.flat[-1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlacao entre features'); plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x='idh', y='dist_ubs_km', hue='demanda', alpha=0.6)
plt.title('IDH x Distancia a UBS por classe de demanda'); plt.show()

## 3. Pré-processamento: split + LabelEncoder

In [ ]:
X = df[features].copy()
le = LabelEncoder()
y = le.fit_transform(df['demanda'].astype(str))
print('classes:', list(le.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED,
)
print('train:', X_train.shape, 'test:', X_test.shape)

## 4. Treino dos 3 modelos com 5-fold estratificado

In [ ]:
modelos = {
    'random_forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)),
    ]),
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial', random_state=SEED)),
    ]),
}
if XGB_OK:
    modelos['xgboost'] = Pipeline([
        ('clf', XGBClassifier(
            n_estimators=400, learning_rate=0.08, max_depth=5,
            objective='multi:softprob', eval_metric='mlogloss',
            random_state=SEED, n_jobs=-1,
        )),
    ])
else:
    modelos['gradient_boosting'] = Pipeline([
        ('clf', GradientBoostingClassifier(n_estimators=300, learning_rate=0.08, max_depth=4, random_state=SEED)),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
resultados = {}
for nome, pipe in modelos.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    resultados[nome] = {'mean': float(scores.mean()), 'std': float(scores.std()), 'scores': scores.tolist()}
    print(f'{nome:>22s}: f1_macro = {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
ranking = pd.DataFrame(resultados).T.sort_values('mean', ascending=False)
ranking

## 5. Seleção do vencedor

In [ ]:
melhor_nome = ranking.index[0]
print('vencedor:', melhor_nome)
pipe_vencedor = modelos[melhor_nome]
pipe_vencedor.fit(X_train, y_train)

## 6. Avaliação no conjunto de teste

In [ ]:
y_pred = pipe_vencedor.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average='macro')
print(f'accuracy = {acc:.4f}')
print(f'f1_macro = {f1m:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=list(le.classes_)))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('previsto'); ax.set_ylabel('real'); ax.set_title('Matriz de confusao - teste')
plt.tight_layout(); plt.show()

## 7. Feature importances (decoy `dia_semana_cadastro` deve ficar no fim)

In [ ]:
clf = pipe_vencedor.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    imps = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
elif hasattr(clf, 'coef_'):
    imps = pd.Series(np.mean(np.abs(clf.coef_), axis=0), index=features).sort_values(ascending=True)
else:
    imps = pd.Series(dtype=float)
imps.plot.barh(figsize=(7,4), title='Importancia das features no modelo vencedor')
plt.tight_layout(); plt.show()
imps[::-1]

## 8. Serialização: `.joblib` + `metrics.json`

In [ ]:
MODELO_PATH = ROOT / 'modelo' / 'modelo_demanda.joblib'
METRICS_PATH = ROOT / 'modelo' / 'metrics.json'

joblib.dump({
    'pipeline': pipe_vencedor,
    'label_encoder': le,
    'features': features,
    'modelo_nome': melhor_nome,
    'versao': f'{melhor_nome}-v1',
}, MODELO_PATH)

metrics = {
    'modelo_escolhido': melhor_nome,
    'versao': f'{melhor_nome}-v1',
    'accuracy': float(acc),
    'f1_macro': float(f1m),
    'f1_macro_cv': resultados,
    'confusion_matrix': cm.tolist(),
    'classes': list(le.classes_),
    'feature_importances_top5': [
        {'feature': f, 'importance': float(v)} for f, v in imps[::-1].head(5).items()
    ],
    'n_treino': int(len(X_train)),
    'n_teste': int(len(X_test)),
    'features': features,
    'xgboost_disponivel': XGB_OK,
    'data_treino': datetime.now(timezone.utc).isoformat(),
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
print('OK ->', MODELO_PATH)
print('OK ->', METRICS_PATH)